In [1]:
import mojo.notebook

In [ ]:
%%mojo

comptime x : Int = 3


def main():
    print("Hello, Mojo!")

Hello, Mojo!



In [4]:
%%file /tmp/mojo_module.mojo
from std.python import PythonObject
from std.python.bindings import PythonModuleBuilder
from std import math
from std.os import abort

@export
def PyInit_mojo_module() abi("C") -> PythonObject:
    try:
        var m = PythonModuleBuilder("mojo_module")
        m.def_function[factorial]("factorial", docstring="Compute n!")
        return m.finalize()
    except e:
        abort(String("error creating Python Mojo module:", e))

def factorial(py_obj: PythonObject) raises -> PythonObject:
    # Raises an exception if `py_obj` is not convertible to a Mojo `Int`.
    var n = Int(py=py_obj)

    return math.factorial(n) + 1

Overwriting /tmp/mojo_module.mojo


In [1]:
import mojo.importer
import sys
# Add the directory containing the file to sys.path
sys.path.append("/tmp")

import mojo_module

print(mojo_module.factorial(5))

121


https://github.com/PyO3/maturin-import-hook?
Holy schnikes. This rules. Every edit is about 10s, which isn't great

In [9]:
%%file /tmp/fast.rs
use pyo3::prelude::*;

#[pyfunction]
fn double(x: usize) -> usize {
    x +1 
}

#[pymodule]
fn fast(m: &Bound<'_, PyModule>) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(double, m)?)?;
    Ok(())
}

Overwriting /tmp/fast.rs


In [2]:
import maturin_import_hook
maturin_import_hook.install()
import sys
sys.path.append("/tmp")

import fast
print(fast.double(21))

43


In [10]:
import importlib
importlib.reload(fast)
print(fast.double(21))

building "fast"
rebuilt and loaded module "fast" in 8.226s


22
